In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

# Load Shared Train/Test Split
train_df = pd.read_csv("../data/processed/train_dataset.csv")
test_df  = pd.read_csv("../data/processed/test_dataset.csv")

X_train = train_df['text'].tolist()
y_train = train_df['label_encoded'].tolist()

X_test = test_df['text'].tolist()
y_test = test_df['label_encoded'].tolist()

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_dict({"text": X_train, "label": y_train})
test_dataset  = Dataset.from_dict({"text": X_test,  "label": y_test})

# Load Tokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization Function
def tokenize_function(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

# Set PyTorch Format
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Load Pretrained Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
)

# Evaluation Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    report = classification_report(labels, preds, output_dict=True)
    return {"accuracy": acc, "precision": report["1"]["precision"], "recall": report["1"]["recall"], "f1": report["1"]["f1-score"]}

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset, 
    compute_metrics=compute_metrics,
)

# Train the Model
trainer.train()

# Predict on Test Set
preds_output = trainer.predict(test_dataset)
pred_labels = np.argmax(preds_output.predictions, axis=1)

# Evaluation
print("Test Accuracy:", accuracy_score(y_test, pred_labels))
print(classification_report(y_test, pred_labels))
print(confusion_matrix(y_test, pred_labels))

# Save Model & Tokenizer
model.save_pretrained("../models/xlm-roberta-fake-news")
tokenizer.save_pretrained("../models/xlm-roberta-fake-news")

# Save Transformer Predictions for Comparison
results_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "transformer_pred": pred_labels
})

results_df.to_csv("../data/processed/transformer_predictions.csv", index=False)